In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    cohen_kappa_score,
    make_scorer,
 )

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


In [12]:
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "data").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "database_non-shows.xlsx"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_PATH = PROCESSED_DIR / "dataset_desbalanceado.csv"

target = 'Appointment Type'

num_cols = [
    'Age', 'Number of Diseases', 'Recent Hospitalization',
    'Number of Medications', 'Hour',
    'Creation to Assignment Interval',
    'Number of Previous Attendance',
    'Number of Previous Non-Attendance'
 ]
cat_cols = ['Sex', 'Insurance Type', 'Day', 'Month']

required_cols = [target, *num_cols, *cat_cols]

def build_processed_dataset(raw_path: Path) -> pd.DataFrame:
    if not raw_path.exists():
        raise FileNotFoundError(f"No encuentro el Excel fuente en: {raw_path}")

    df_raw = pd.read_excel(raw_path)
    missing = [c for c in required_cols if c not in df_raw.columns]
    if missing:
        raise ValueError("Faltan columnas requeridas en el Excel: " + ", ".join(missing))

    df_clean = df_raw[required_cols].copy()
    df_clean = df_clean.dropna()

    # Reglas básicas (consistentes con 01_Exploración.ipynb)
    condiciones = {
        'Appointment Type': ~df_clean['Appointment Type'].isin([0, 1]),
        'Age': (df_clean['Age'] < 18) | (df_clean['Age'] > 120),
        'Sex': ~df_clean['Sex'].isin([0, 1, 2]),
        'Insurance Type': ~df_clean['Insurance Type'].isin(range(0, 9)),
        'Number of Diseases': (df_clean['Number of Diseases'] < 0),
        'Recent Hospitalization': (df_clean['Recent Hospitalization'] < 0),
        'Number of Medications': (df_clean['Number of Medications'] < 0),
        'Hour': (df_clean['Hour'] < 0) | (df_clean['Hour'] > 23),
        'Day': ~df_clean['Day'].between(0, 6),
        'Month': ~df_clean['Month'].between(1, 12),
        'Creation to Assignment Interval': (df_clean['Creation to Assignment Interval'] < 0),
        'Number of Previous Attendance': (df_clean['Number of Previous Attendance'] < 0),
        'Number of Previous Non-Attendance': (df_clean['Number of Previous Non-Attendance'] < 0),
    }
    mask_invalid = pd.Series(False, index=df_clean.index)
    for cond in condiciones.values():
        mask_invalid = mask_invalid | cond
    df_clean = df_clean.loc[~mask_invalid].copy()

    return df_clean

def load_processed_dataset(processed_path: Path = PROCESSED_PATH) -> pd.DataFrame:
    if processed_path.exists():
        return pd.read_csv(processed_path)

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    df_clean = build_processed_dataset(RAW_PATH)
    df_clean.to_csv(processed_path, index=False)
    return df_clean

def balance_dataset_smote(X_in: pd.DataFrame, y_in: pd.Series, random_state: int = 42):
    """Balanceo simple con SMOTE (ojo: úsalo SOLO sobre entrenamiento para evitar data leakage)."""
    smote = SMOTE(random_state=random_state)
    return smote.fit_resample(X_in, y_in)

# Cargar o construir dataset procesado
df = load_processed_dataset()

X = df[num_cols + cat_cols]
y = df[target]

print("Distribución de clases (original):")
print(y.value_counts())

# Preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', 'passthrough', cat_cols)
    ]
)

Distribución de clases (original):
Appointment Type
0    12115
1     6276
Name: count, dtype: int64


# Decision Tree

In [4]:
df = load_processed_dataset()

target = 'Appointment Type'

num_cols = [
    'Age', 'Number of Diseases', 'Recent Hospitalization',
    'Number of Medications', 'Hour',
    'Creation to Assignment Interval',
    'Number of Previous Attendance',
    'Number of Previous Non-Attendance'
 ]

cat_cols = ['Sex', 'Insurance Type', 'Day', 'Month']

X = df[num_cols + cat_cols]
y = df[target]

# Preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', 'passthrough', cat_cols)
    ]
)

# Scorers enfocados en reducir "predije 0 pero era 1" (FN de la clase 1)
recall_pos1 = make_scorer(recall_score, pos_label=1)
f1_pos1 = make_scorer(f1_score, pos_label=1)

# Pipeline con SMOTE (usar ImbPipeline de imblearn)
pipeline = ImbPipeline(steps=[
    ('preprocess', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('select', SelectKBest(score_func=f_classif)),
    ('model', DecisionTreeClassifier(random_state=42, class_weight='balanced'))
])

param_grid = {
    'select__k': [10, 12, 'all'],

    # Nota: antes estaba muy restrictivo (leaf/split grandes) y tendía a subajustar.
    'model__max_depth': [6, 10, 15, None],
    'model__min_samples_split': [2, 10, 50, 100],
    'model__min_samples_leaf': [1, 5, 10, 20],
    'model__criterion': ['gini', 'entropy']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Iniciando GridSearchCV (Decision Tree) optimizando F1/Recall de la clase 1...")
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring={
        'accuracy': 'accuracy',
        'f1_1': f1_pos1,
        'recall_1': recall_pos1,
        'f1_weighted': 'f1_weighted',
    },
    refit='f1_1',
    n_jobs=-1,
    verbose=1
)

grid.fit(X, y)

print("\nMejores parámetros:")
print(grid.best_params_)
best_idx = grid.best_index_
print(f"Mejor F1(clase=1) en CV: {grid.best_score_:.4f}")
print(f"Recall(clase=1) en CV: {grid.cv_results_['mean_test_recall_1'][best_idx]:.4f}")
print(f"Accuracy en CV:        {grid.cv_results_['mean_test_accuracy'][best_idx]:.4f}")

best_model = grid.best_estimator_

print("\n===== ENTRENAMIENTO REPETIDO =====")

N = 10
results = []

for seed in range(N):
    model_rep = ImbPipeline(steps=[
        ('preprocess', preprocessor),
        ('smote', SMOTE(random_state=seed)),
        ('select', SelectKBest(
            score_func=f_classif,
            k=grid.best_params_['select__k']
        )),
        ('model', DecisionTreeClassifier(
            random_state=seed,
            class_weight='balanced',
            max_depth=grid.best_params_['model__max_depth'],
            min_samples_split=grid.best_params_['model__min_samples_split'],
            min_samples_leaf=grid.best_params_['model__min_samples_leaf'],
            criterion=grid.best_params_['model__criterion']
        ))
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    model_rep.fit(X_train, y_train)
    y_pred = model_rep.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1w = f1_score(y_test, y_pred, average='weighted')
    f1m = f1_score(y_test, y_pred, average='macro')
    f1_1 = f1_score(y_test, y_pred, pos_label=1)
    rec_1 = recall_score(y_test, y_pred, pos_label=1)
    kappa = cohen_kappa_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    fn_1 = int(cm[1, 0])
    fp_1 = int(cm[0, 1])

    results.append((seed, f1_1, acc, f1w, f1m, rec_1, kappa, model_rep, y_test, y_pred))
    print(
        f"Seed {seed} → Acc: {acc:.4f} | Kappa: {kappa:.4f} | F1(1): {f1_1:.4f} | Recall(1): {rec_1:.4f} | "
        f"F1(w): {f1w:.4f} | FN(1→0): {fn_1} | FP(0→1): {fp_1}"
    )

# Elegimos el mejor por F1 de la clase 1 (alineado a reducir FN de clase 1)
best_seed, best_f1_1, best_acc, best_f1w, best_f1m, best_rec_1, best_kappa, best_model_final, y_test_final, y_pred_final = max(
    results, key=lambda x: x[1]
 )

print("\n" + "="*50)
print("MEJOR MODELO FINAL (Decision Tree)")
print("="*50)
print(f"Seed: {best_seed}")
print(f"Accuracy: {best_acc:.4f}")
print(f"Kappa: {best_kappa:.4f}")
print(f"F1 weighted: {best_f1w:.4f}")
print(f"F1 macro: {best_f1m:.4f}")
print(f"F1 (clase=1): {best_f1_1:.4f}")
print(f"Recall (clase=1): {best_rec_1:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test_final, y_pred_final, digits=3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_final, y_pred_final))

print("\n" + "="*50)
print("DISTRIBUCIÓN DE CLASES")
print("="*50)
print("\nOriginal:")
print(y.value_counts())
print(f"\nProporción: {y.value_counts(normalize=True)}")

Iniciando GridSearchCV (Decision Tree) optimizando F1/Recall de la clase 1...
Fitting 5 folds for each of 384 candidates, totalling 1920 fits

Mejores parámetros:
{'model__criterion': 'entropy', 'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 100, 'select__k': 12}
Mejor F1(clase=1) en CV: 0.6908
Recall(clase=1) en CV: 0.8043
Accuracy en CV:        0.7542

===== ENTRENAMIENTO REPETIDO =====
Seed 0 → Acc: 0.7521 | Kappa: 0.4864 | F1(1): 0.6849 | Recall(1): 0.7896 | F1(w): 0.7579 | FN(1→0): 264 | FP(0→1): 648
Seed 1 → Acc: 0.7633 | Kappa: 0.5048 | F1(1): 0.6934 | Recall(1): 0.7849 | F1(w): 0.7684 | FN(1→0): 270 | FP(0→1): 601
Seed 2 → Acc: 0.7546 | Kappa: 0.4952 | F1(1): 0.6923 | Recall(1): 0.8096 | F1(w): 0.7605 | FN(1→0): 239 | FP(0→1): 664
Seed 3 → Acc: 0.7486 | Kappa: 0.4845 | F1(1): 0.6868 | Recall(1): 0.8080 | F1(w): 0.7548 | FN(1→0): 241 | FP(0→1): 684
Seed 4 → Acc: 0.7510 | Kappa: 0.4812 | F1(1): 0.6799 | Recall(1): 0.7753 | F1(w): 0.7566 | FN(1→0

# Random Forest

In [5]:
df = load_processed_dataset()

target = 'Appointment Type'

num_cols = [
    'Age', 'Number of Diseases', 'Recent Hospitalization',
    'Number of Medications', 'Hour',
    'Creation to Assignment Interval',
    'Number of Previous Attendance',
    'Number of Previous Non-Attendance'
 ]

cat_cols = ['Sex', 'Insurance Type', 'Day', 'Month']

X = df[num_cols + cat_cols]
y = df[target]

# Preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', 'passthrough', cat_cols)
    ]
)

# Scorers enfocados en reducir "predije 0 pero era 1" (FN de la clase 1)
recall_pos1 = make_scorer(recall_score, pos_label=1)
f1_pos1 = make_scorer(f1_score, pos_label=1)

# Pipeline con SMOTE y Random Forest
pipeline = ImbPipeline(steps=[
    ('preprocess', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('select', SelectKBest(score_func=f_classif)),
    ('model', RandomForestClassifier(
        random_state=42,
        class_weight='balanced_subsample',
    ))
])

# Grid de parámetros para Random Forest (mantengo tamaño similar al actual)
param_grid = {
    'select__k': [10, 12, 'all'],

    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 15, None],
    'model__min_samples_split': [10, 20],
    'model__min_samples_leaf': [1, 5, 10],
    'model__max_features': ['sqrt', 'log2'],
    'model__criterion': ['gini', 'entropy']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Iniciando GridSearchCV (Random Forest) optimizando F1/Recall de la clase 1...")
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring={
        'accuracy': 'accuracy',
        'f1_1': f1_pos1,
        'recall_1': recall_pos1,
        'f1_weighted': 'f1_weighted',
    },
    refit='f1_1',
    n_jobs=-1,
    verbose=1
)

grid.fit(X, y)

print("\n" + "="*60)
print("MEJORES PARÁMETROS ENCONTRADOS")
print("="*60)
for param, value in grid.best_params_.items():
    print(f"{param}: {value}")

best_idx = grid.best_index_
print(f"\nMejor F1(clase=1) en CV: {grid.best_score_:.4f}")
print(f"Recall(clase=1) en CV: {grid.cv_results_['mean_test_recall_1'][best_idx]:.4f}")
print(f"Accuracy en CV:        {grid.cv_results_['mean_test_accuracy'][best_idx]:.4f}")

best_model = grid.best_estimator_

print("\n" + "="*60)
print("ENTRENAMIENTO REPETIDO CON DIFERENTES SEEDS")
print("\n" + "="*60)

N = 4
results = []

for seed in range(N):
    model_rep = ImbPipeline(steps=[
        ('preprocess', preprocessor),
        ('smote', SMOTE(random_state=seed)),
        ('select', SelectKBest(
            score_func=f_classif,
            k=grid.best_params_['select__k']
        )),
        ('model', RandomForestClassifier(
            random_state=seed,
            class_weight='balanced_subsample',
            n_estimators=grid.best_params_['model__n_estimators'],
            max_depth=grid.best_params_['model__max_depth'],
            min_samples_split=grid.best_params_['model__min_samples_split'],
            min_samples_leaf=grid.best_params_['model__min_samples_leaf'],
            max_features=grid.best_params_['model__max_features'],
            criterion=grid.best_params_['model__criterion']
        ))
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    model_rep.fit(X_train, y_train)
    y_pred = model_rep.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1w = f1_score(y_test, y_pred, average='weighted')
    f1m = f1_score(y_test, y_pred, average='macro')
    f1_1 = f1_score(y_test, y_pred, pos_label=1)
    rec_1 = recall_score(y_test, y_pred, pos_label=1)
    kappa = cohen_kappa_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    fn_1 = int(cm[1, 0])
    fp_1 = int(cm[0, 1])

    results.append((seed, f1_1, acc, f1w, f1m, rec_1, kappa, model_rep, y_test, y_pred))
    print(
        f"Seed {seed:2d} → Acc: {acc:.4f} | Kappa: {kappa:.4f} | F1(1): {f1_1:.4f} | Recall(1): {rec_1:.4f} | "
        f"F1(w): {f1w:.4f} | FN(1→0): {fn_1} | FP(0→1): {fp_1}"
    )

best_seed, best_f1_1, best_acc, best_f1w, best_f1m, best_rec_1, best_kappa, best_model_final, y_test_final, y_pred_final = max(
    results, key=lambda x: x[1]
 )

print("\n" + "="*60)
print("MEJOR MODELO FINAL (RANDOM FOREST)")
print("="*60)
print(f"Seed: {best_seed}")
print(f"Accuracy: {best_acc:.4f}")
print(f"Kappa: {best_kappa:.4f}")
print(f"F1 weighted: {best_f1w:.4f}")
print(f"F1 macro: {best_f1m:.4f}")
print(f"F1 (clase=1): {best_f1_1:.4f}")
print(f"Recall (clase=1): {best_rec_1:.4f}")

print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test_final, y_pred_final, digits=3))

print("\n" + "="*60)
print("CONFUSION MATRIX")
print("="*60)
print(confusion_matrix(y_test_final, y_pred_final))

print("\n" + "="*60)
print("IMPORTANCIA DE CARACTERÍSTICAS (TOP 10)")
print("="*60)

rf_model = best_model_final.named_steps['model']
feature_names = num_cols + cat_cols

importances = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(importances.head(10).to_string(index=False))

print("\n" + "="*60)
print("ESTADÍSTICAS DE LOS ENTRENAMIENTOS")
print("="*60)
accuracies = [r[2] for r in results]
f1_1_scores = [r[1] for r in results]
rec_1_scores = [r[5] for r in results]
kappas = [r[6] for r in results]

print(f"Accuracy promedio:    {np.mean(accuracies):.4f}")
print(f"Accuracy std:         {np.std(accuracies):.4f}")
print(f"F1(clase=1) prom:     {np.mean(f1_1_scores):.4f}")
print(f"F1(clase=1) std:      {np.std(f1_1_scores):.4f}")
print(f"Recall(clase=1) prom: {np.mean(rec_1_scores):.4f}")
print(f"Recall(clase=1) std:  {np.std(rec_1_scores):.4f}")
print(f"Kappa prom:           {np.mean(kappas):.4f}")
print(f"Kappa std:            {np.std(kappas):.4f}")

print("\n" + "="*60)
print("DISTRIBUCIÓN DE CLASES")
print("="*60)
print("\nOriginal:")
print(y.value_counts())
print(f"\nProporción:")
print(y.value_counts(normalize=True).round(3))

Iniciando GridSearchCV (Random Forest) optimizando F1/Recall de la clase 1...
Fitting 5 folds for each of 432 candidates, totalling 2160 fits


c:\Users\samue\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



MEJORES PARÁMETROS ENCONTRADOS
model__criterion: entropy
model__max_depth: None
model__max_features: sqrt
model__min_samples_leaf: 1
model__min_samples_split: 20
model__n_estimators: 200
select__k: 12

Mejor F1(clase=1) en CV: 0.7115
Recall(clase=1) en CV: 0.7071
Accuracy en CV:        0.8043

ENTRENAMIENTO REPETIDO CON DIFERENTES SEEDS

Seed  0 → Acc: 0.7918 | Kappa: 0.5347 | F1(1): 0.6919 | Recall(1): 0.6853 | F1(w): 0.7913 | FN(1→0): 395 | FP(0→1): 371
Seed  1 → Acc: 0.8103 | Kappa: 0.5761 | F1(1): 0.7195 | Recall(1): 0.7131 | F1(w): 0.8099 | FN(1→0): 360 | FP(0→1): 338
Seed  2 → Acc: 0.8013 | Kappa: 0.5565 | F1(1): 0.7068 | Recall(1): 0.7020 | F1(w): 0.8010 | FN(1→0): 374 | FP(0→1): 357
Seed  3 → Acc: 0.7970 | Kappa: 0.5479 | F1(1): 0.7018 | Recall(1): 0.7004 | F1(w): 0.7969 | FN(1→0): 376 | FP(0→1): 371

MEJOR MODELO FINAL (RANDOM FOREST)
Seed: 1
Accuracy: 0.8103
Kappa: 0.5761
F1 weighted: 0.8099
F1 macro: 0.7881
F1 (clase=1): 0.7195
Recall (clase=1): 0.7131

CLASSIFICATION REPOR

# SVM

In [ ]:
df = load_processed_dataset()

target = 'Appointment Type'

num_cols = [
    'Age', 'Number of Diseases', 'Recent Hospitalization',
    'Number of Medications', 'Hour',
    'Creation to Assignment Interval',
    'Number of Previous Attendance',
    'Number of Previous Non-Attendance'
 ]

cat_cols = ['Sex', 'Insurance Type', 'Day', 'Month']

X = df[num_cols + cat_cols]
y = df[target]

# Preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', 'passthrough', cat_cols)
    ]
)

# Scorers enfocados en reducir "predije 0 pero era 1" (FN de la clase 1)
recall_pos1 = make_scorer(recall_score, pos_label=1)
f1_pos1 = make_scorer(f1_score, pos_label=1)

# Usamos ImbPipeline para que el GridSearch y el entrenamiento repetido sean consistentes (con SMOTE)
pipeline = ImbPipeline(steps=[
    ('preprocess', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('select', SelectKBest(score_func=f_classif)),
    ('model', SVC(probability=True, random_state=42))
 ])

param_grid = {
    'select__k': [10, 12, 'all'],

    'model__C': [0.1, 1, 10],          # Regularización
    'model__kernel': ['linear', 'rbf'],
    'model__gamma': ['scale', 'auto'],  # Solo afecta rbf
    'model__class_weight': [None, 'balanced'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Iniciando GridSearchCV (SVM) optimizando F1/Recall de la clase 1...")
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring={
        'accuracy': 'accuracy',
        'f1_1': f1_pos1,
        'recall_1': recall_pos1,
        'f1_weighted': 'f1_weighted',
    },
    refit='f1_1',
    n_jobs=-1,
    verbose=1
)

grid.fit(X, y)

print("\nMejores parámetros:", grid.best_params_)
best_idx = grid.best_index_
print(f"Mejor F1(clase=1) en CV: {grid.best_score_:.4f}")
print(f"Recall(clase=1) en CV: {grid.cv_results_['mean_test_recall_1'][best_idx]:.4f}")
print(f"Accuracy en CV:        {grid.cv_results_['mean_test_accuracy'][best_idx]:.4f}")

best_model = grid.best_estimator_

results = []
N = 10

for seed in range(N):
    model_rep = ImbPipeline(steps=[
        ('preprocess', preprocessor),
        ('smote', SMOTE(random_state=seed)),
        ('select', SelectKBest(
            score_func=f_classif,
            k=grid.best_params_['select__k']
        )),
        ('model', SVC(
            probability=True,
            random_state=seed,
            C=grid.best_params_['model__C'],
            kernel=grid.best_params_['model__kernel'],
            gamma=grid.best_params_['model__gamma'],
            class_weight=grid.best_params_['model__class_weight'],
        ))
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    model_rep.fit(X_train, y_train)
    y_pred = model_rep.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1w = f1_score(y_test, y_pred, average='weighted')
    f1m = f1_score(y_test, y_pred, average='macro')
    f1_1 = f1_score(y_test, y_pred, pos_label=1)
    rec_1 = recall_score(y_test, y_pred, pos_label=1)
    kappa = cohen_kappa_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    fn_1 = int(cm[1, 0])
    fp_1 = int(cm[0, 1])

    results.append((seed, f1_1, acc, f1w, f1m, rec_1, kappa, model_rep, y_test, y_pred))
    print(
        f"Seed {seed} → Acc: {acc:.4f} | Kappa: {kappa:.4f} | F1(1): {f1_1:.4f} | Recall(1): {rec_1:.4f} | "
        f"F1(w): {f1w:.4f} | FN(1→0): {fn_1} | FP(0→1): {fp_1}"
    )

best_seed, best_f1_1, best_acc, best_f1w, best_f1m, best_rec_1, best_kappa, best_model_final, y_test_final, y_pred_final = max(
    results, key=lambda x: x[1]
 )

print("\nMEJOR MODELO FINAL (SVM)")
print(f"Seed: {best_seed}")
print(f"Accuracy: {best_acc:.4f}")
print(f"Kappa: {best_kappa:.4f}")
print(f"F1 weighted: {best_f1w:.4f}")
print(f"F1 macro: {best_f1m:.4f}")
print(f"F1 (clase=1): {best_f1_1:.4f}")
print(f"Recall (clase=1): {best_rec_1:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test_final, y_pred_final, digits=3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_final, y_pred_final))


Iniciando GridSearchCV (SVM) optimizando F1/Recall de la clase 1...
Fitting 5 folds for each of 72 candidates, totalling 360 fits


# Regresion logistica

In [17]:
# Logistic Regression (ajustada para NO colapsar a FN masivos al minimizar FP)
df = load_processed_dataset()

target = 'Appointment Type'  # 0 = asistió, 1 = no asistió

num_cols = [
    'Age', 'Number of Diseases', 'Recent Hospitalization',
    'Number of Medications', 'Hour',
    'Creation to Assignment Interval',
    'Number of Previous Attendance',
    'Number of Previous Non-Attendance'
 ]
cat_cols = ['Sex', 'Insurance Type', 'Day', 'Month']

X = df[num_cols + cat_cols]
y = df[target]

# Objetivo: reducir FP(0→1) sin caer en la "trampa" de predecir casi todo 0.
# Nota importante: si tu modelo NO logra acc>=0.80 en validación, ese requisito es inalcanzable para LR con estas features;
# en ese caso, el umbral se elige cerca del máximo accuracy (y recién ahí se reduce FP).
USE_SMOTE = True
MIN_ACC = 0.80              # tu requisito
MIN_RECALL_1 = 0.25         # evita soluciones con recall(1) casi cero
ACC_TOL = 0.003             # tolerancia para "casi el mejor accuracy" cuando MIN_ACC es imposible
FP_WEIGHT = 1.0             # costo por FP(0→1)
FN_WEIGHT = 0.02            # costo por FN(1→0) (sube esto si quieres menos FN)
thresholds = np.linspace(0.05, 0.95, 37)

baseline_acc_all_zero = float((y == 0).mean())
print(f"Baseline si siempre predigo 0 (asistió): Acc={baseline_acc_all_zero:.4f}")
print(f"Distribución clases: {y.value_counts().to_dict()}")

def make_ohe_dense():
    # Compatibilidad scikit-learn: sparse_output (nuevo) vs sparse (antiguo)
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

ohe = make_ohe_dense()

# OneHot denso permite usar SMOTE después del preprocesado (si USE_SMOTE=True)
preprocessor_lr = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', ohe, cat_cols),
    ],
    remainder='drop'
 )

precision_pos1 = make_scorer(precision_score, pos_label=1, zero_division=0)
recall_pos1 = make_scorer(recall_score, pos_label=1)
f1_pos1 = make_scorer(f1_score, pos_label=1, zero_division=0)

def build_lr_pipeline(random_state: int, use_smote: bool, model_kwargs: dict):
    # Usamos saga + elasticnet porque en tu sklearn la forma antigua (penalty) está deprecated.
    lr = LogisticRegression(
        max_iter=10000,
        solver='saga',
        penalty='elasticnet',
        n_jobs=-1,
        random_state=random_state,
        **model_kwargs,
    )
    if use_smote:
        return ImbPipeline(steps=[
            ('preprocess', preprocessor_lr),
            ('smote', SMOTE(random_state=random_state)),
            ('model', lr),
        ])
    return Pipeline(steps=[
        ('preprocess', preprocessor_lr),
        ('model', lr),
    ])

# Grid (regularización + mezcla L1/L2 vía l1_ratio)
base_pipeline = build_lr_pipeline(
    random_state=42,
    use_smote=USE_SMOTE,
    model_kwargs={}
 )

param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__l1_ratio': [0.0, 0.5, 1.0],  # 0=l2, 1=l1
    'model__class_weight': [None, 'balanced'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\nIniciando GridSearchCV (Logistic Regression) refit=accuracy...")
grid = GridSearchCV(
    base_pipeline,
    param_grid,
    cv=cv,
    scoring={
        'accuracy': 'accuracy',
        'precision_1': precision_pos1,
        'recall_1': recall_pos1,
        'f1_1': f1_pos1,
        'f1_weighted': 'f1_weighted',
    },
    refit='accuracy',
    n_jobs=-1,
    verbose=1,
 )

grid.fit(X, y)

best_idx = grid.best_index_
print("\nMejores parámetros:")
print(grid.best_params_)
print(f"Accuracy en CV:        {grid.cv_results_['mean_test_accuracy'][best_idx]:.4f}")
print(f"Precisión(clase=1) CV: {grid.cv_results_['mean_test_precision_1'][best_idx]:.4f}")
print(f"Recall(clase=1) CV:    {grid.cv_results_['mean_test_recall_1'][best_idx]:.4f}")
print(f"F1(clase=1) CV:        {grid.cv_results_['mean_test_f1_1'][best_idx]:.4f}")

best_model_kwargs = {
    'C': grid.best_params_['model__C'],
    'l1_ratio': grid.best_params_['model__l1_ratio'],
    'class_weight': grid.best_params_['model__class_weight'],
}

print("\n===== ENTRENAMIENTO REPETIDO + TUNING DE UMBRAL (FP vs FN) =====")
print(f"Criterio umbral: MIN_ACC={MIN_ACC}, MIN_RECALL_1={MIN_RECALL_1}, ACC_TOL={ACC_TOL}, FP_WEIGHT={FP_WEIGHT}, FN_WEIGHT={FN_WEIGHT}\n")

def eval_thresholds(y_true, proba_1, thresholds_arr):
    rows = []
    for t in thresholds_arr:
        y_pred = (proba_1 >= t).astype(int)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        fp = int(cm[0, 1])  # predijo 1 pero era 0
        fn = int(cm[1, 0])  # predijo 0 pero era 1
        acc = float(accuracy_score(y_true, y_pred))
        prec1 = float(precision_score(y_true, y_pred, pos_label=1, zero_division=0))
        rec1 = float(recall_score(y_true, y_pred, pos_label=1))
        f11 = float(f1_score(y_true, y_pred, pos_label=1, zero_division=0))
        pred_pos = int((y_pred == 1).sum())
        cost = float(FP_WEIGHT * fp + FN_WEIGHT * fn)
        rows.append({
            't': float(t),
            'fp': fp,
            'fn': fn,
            'acc': acc,
            'prec1': prec1,
            'rec1': rec1,
            'f11': f11,
            'pred_pos': pred_pos,
            'cost': cost,
        })
    return rows

def pick_threshold(rows, min_acc, min_recall_1):
    max_acc = max(r['acc'] for r in rows)
    # 1) Intento estricto: accuracy>=min_acc y recall(1)>=min_recall_1
    feasible = [r for r in rows if (r['acc'] >= min_acc and r['rec1'] >= min_recall_1)]
    if feasible:
        best = min(feasible, key=lambda r: (r['fp'], r['cost'], -r['acc']))
        return best, 'VAL_OK(acc&recall)', True, max_acc
    # 2) Relajo recall: solo accuracy>=min_acc
    feasible_acc = [r for r in rows if r['acc'] >= min_acc]
    if feasible_acc:
        best = min(feasible_acc, key=lambda r: (r['fp'], r['cost'], -r['acc']))
        return best, 'VAL_OK(acc_only)', True, max_acc
    # 3) MIN_ACC imposible: primero busco umbrales con accuracy casi máxima, y dentro reduzco FP/costo
    near_best = [r for r in rows if r['acc'] >= (max_acc - ACC_TOL) and r['rec1'] >= min_recall_1]
    if not near_best:
        near_best = [r for r in rows if r['acc'] >= (max_acc - ACC_TOL)]
    best = min(near_best, key=lambda r: (r['fp'], r['cost'], -r['rec1']))
    return best, f'VAL_NO(max_acc={max_acc:.3f})', False, max_acc

def select_best(results, min_acc):
    candidates = [r for r in results if r[3] >= min_acc]
    pool = candidates if candidates else results
    # Prioridad: menos FP, luego menos costo, luego mayor accuracy
    return min(pool, key=lambda r: (r[1], (FP_WEIGHT * r[1] + FN_WEIGHT * r[2]), -r[3]))

N = 10
results = []

for seed in range(N):
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.25, random_state=seed, stratify=y_train_full
    )
    model_inner = build_lr_pipeline(
        random_state=seed,
        use_smote=USE_SMOTE,
        model_kwargs=best_model_kwargs
    )
    model_inner.fit(X_train, y_train)
    proba_val = model_inner.predict_proba(X_val)[:, 1]
    rows = eval_thresholds(y_val, proba_val, thresholds)
    choice, val_note, feasible_found, max_acc = pick_threshold(rows, MIN_ACC, MIN_RECALL_1)
    best_t = choice['t']

    model_final = build_lr_pipeline(
        random_state=seed,
        use_smote=USE_SMOTE,
        model_kwargs=best_model_kwargs
    )
    model_final.fit(X_train_full, y_train_full)
    proba_test = model_final.predict_proba(X_test)[:, 1]
    y_pred = (proba_test >= best_t).astype(int)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1])
    acc = float(accuracy_score(y_test, y_pred))
    kappa = float(cohen_kappa_score(y_test, y_pred))
    prec1 = float(precision_score(y_test, y_pred, pos_label=1, zero_division=0))
    rec1 = float(recall_score(y_test, y_pred, pos_label=1))
    f11 = float(f1_score(y_test, y_pred, pos_label=1, zero_division=0))
    f1w = float(f1_score(y_test, y_pred, average='weighted'))
    pred_pos = int((y_pred == 1).sum())

    results.append((seed, fp, fn, acc, kappa, prec1, rec1, f11, f1w, best_t, pred_pos, val_note, model_final, y_test, y_pred))
    acc_note = "ACC_OK" if acc >= MIN_ACC else "ACC_NO"
    print(
        f"Seed {seed} → {val_note} | Thr: {best_t:.2f} | {acc_note} Acc: {acc:.4f} | Kappa: {kappa:.4f} | "
        f"Prec(1): {prec1:.4f} | Recall(1): {rec1:.4f} | FP(0→1): {fp} | FN(1→0): {fn} | Pred(1): {pred_pos}"
    )

best = select_best(results, MIN_ACC)
best_seed, best_fp, best_fn, best_acc, best_kappa, best_prec1, best_rec1, best_f11, best_f1w, best_thr, best_pred_pos, best_val_note, best_model_final, y_test_final, y_pred_final = best

print("\n" + "="*60)
print("MEJOR MODELO FINAL (Logistic Regression + Threshold)")
print("="*60)
print(f"Seed: {best_seed}")
print(f"VAL:  {best_val_note}")
print(f"Threshold: {best_thr:.2f}")
print(f"Accuracy: {best_acc:.4f}")
print(f"Kappa: {best_kappa:.4f}")
print(f"Precision (clase=1): {best_prec1:.4f}")
print(f"Recall (clase=1):    {best_rec1:.4f}")
print(f"F1 (clase=1):        {best_f11:.4f}")
print(f"F1 weighted:         {best_f1w:.4f}")
print(f"FP(0→1): {best_fp}")
print(f"FN(1→0): {best_fn}")
print(f"Predicciones clase 1: {best_pred_pos}")

print("\nClassification Report:\n")
print(classification_report(y_test_final, y_pred_final, digits=3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_final, y_pred_final))

Baseline si siempre predigo 0 (asistió): Acc=0.6587
Distribución clases: {0: 12115, 1: 6276}

Iniciando GridSearchCV (Logistic Regression) refit=accuracy...
Fitting 5 folds for each of 30 candidates, totalling 150 fits


c:\Users\samue\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\samue\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



Mejores parámetros:
{'model__C': 10, 'model__class_weight': None, 'model__l1_ratio': 1.0}
Accuracy en CV:        0.7229
Precisión(clase=1) CV: 0.5803
Recall(clase=1) CV:    0.6794
F1(clase=1) CV:        0.6259

===== ENTRENAMIENTO REPETIDO + TUNING DE UMBRAL (FP vs FN) =====
Criterio umbral: MIN_ACC=0.8, MIN_RECALL_1=0.25, ACC_TOL=0.003, FP_WEIGHT=1.0, FN_WEIGHT=0.02



c:\Users\samue\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\samue\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\samue\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 

Seed 0 → VAL_NO(max_acc=0.754) | Thr: 0.65 | ACC_NO Acc: 0.7450 | Kappa: 0.3609 | Prec(1): 0.7369 | Recall(1): 0.3928 | FP(0→1): 176 | FN(1→0): 762 | Pred(1): 669


...

Seed 7 → VAL_NO(max_acc=0.742) | Thr: 0.70 | ACC_NO Acc: 0.7442 | Kappa: 0.3421 | Prec(1): 0.7844 | Recall(1): 0.3450 | FP(0→1): 119 | FN(1→0): 822 | Pred(1): 552


c:\Users\samue\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\samue\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\samue\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 

Seed 8 → VAL_NO(max_acc=0.753) | Thr: 0.65 | ACC_NO Acc: 0.7559 | Kappa: 0.3944 | Prec(1): 0.7490 | Recall(1): 0.4279 | FP(0→1): 180 | FN(1→0): 718 | Pred(1): 717


c:\Users\samue\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\samue\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\samue\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 

Seed 9 → VAL_NO(max_acc=0.742) | Thr: 0.62 | ACC_NO Acc: 0.7502 | Kappa: 0.3903 | Prec(1): 0.7116 | Recall(1): 0.4502 | FP(0→1): 229 | FN(1→0): 690 | Pred(1): 794

MEJOR MODELO FINAL (Logistic Regression + Threshold)
Seed: 7
VAL:  VAL_NO(max_acc=0.742)
Threshold: 0.70
Accuracy: 0.7442
Kappa: 0.3421
Precision (clase=1): 0.7844
Recall (clase=1):    0.3450
F1 (clase=1):        0.4792
F1 weighted:         0.7107
FP(0→1): 119
FN(1→0): 822
Predicciones clase 1: 552

Classification Report:

              precision    recall  f1-score   support

           0      0.737     0.951     0.830      2424
           1      0.784     0.345     0.479      1255

    accuracy                          0.744      3679
   macro avg      0.761     0.648     0.655      3679
weighted avg      0.753     0.744     0.711      3679


Confusion Matrix:
[[2305  119]
 [ 822  433]]


# Extra Trees (mejor accuracy + control de FP con umbral)

In [15]:
# ExtraTreesClassifier + tuning de umbral para minimizar FP(0→1) con accuracy >= 0.80
df = load_processed_dataset()

target = 'Appointment Type'  # 0 = asistió, 1 = no asistió

num_cols = [
    'Age', 'Number of Diseases', 'Recent Hospitalization',
    'Number of Medications', 'Hour',
    'Creation to Assignment Interval',
    'Number of Previous Attendance',
    'Number of Previous Non-Attendance'
 ]
cat_cols = ['Sex', 'Insurance Type', 'Day', 'Month']

X = df[num_cols + cat_cols]
y = df[target]

preprocessor_et = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', 'passthrough', cat_cols),
    ]
)

precision_pos1 = make_scorer(precision_score, pos_label=1, zero_division=0)

pipeline = Pipeline(steps=[
    ('preprocess', preprocessor_et),
    ('model', ExtraTreesClassifier(random_state=42, n_jobs=-1))
])

param_grid = {
    'model__n_estimators': [300],
    'model__max_depth': [None, 20],
    'model__min_samples_leaf': [1, 5],
    'model__max_features': ['sqrt', 'log2'],
    'model__class_weight': [None, 'balanced_subsample'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Iniciando GridSearchCV (ExtraTrees) optimizando ACC (y reportando precisión clase 1)...")
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring={
        'accuracy': 'accuracy',
        'precision_1': precision_pos1,
        'f1_weighted': 'f1_weighted',
    },
    refit='accuracy',
    n_jobs=-1,
    verbose=1,
 )

grid.fit(X, y)

best_idx = grid.best_index_
print("\nMejores parámetros:")
print(grid.best_params_)
print(f"Accuracy en CV:        {grid.cv_results_['mean_test_accuracy'][best_idx]:.4f}")
print(f"Precisión(clase=1) CV: {grid.cv_results_['mean_test_precision_1'][best_idx]:.4f}")

print("\n===== ENTRENAMIENTO REPETIDO + TUNING DE UMBRAL (min FP con acc>=0.80) =====")

def pick_threshold_min_fp(y_true, proba_1, thresholds, min_acc=0.80):
    """
    Elige umbral para reducir FP(0→1) manteniendo acc>=min_acc si es posible.
    Si no existe ningún umbral que cumpla min_acc, elige el umbral con mayor accuracy
    (y, a igualdad, menor FP).
    """
    rows = []
    for t in thresholds:
        y_pred = (proba_1 >= t).astype(int)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        fp = int(cm[0, 1])  # predijo 1 pero era 0
        fn = int(cm[1, 0])  # predijo 0 pero era 1
        acc = accuracy_score(y_true, y_pred)
        prec1 = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
        rec1 = recall_score(y_true, y_pred, pos_label=1)
        f11 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
        rows.append((t, fp, fn, acc, prec1, rec1, f11))

    max_acc = max(r[3] for r in rows)
    feasible = [r for r in rows if r[3] >= min_acc]
    feasible_found = len(feasible) > 0
    pool = feasible if feasible_found else rows

    # Dentro del pool: minimizar FP, luego maximizar accuracy, luego maximizar precision(1)
    t, fp, fn, acc, prec1, rec1, f11 = min(pool, key=lambda r: (r[1], -r[3], -r[4]))
    return {
        'threshold': float(t),
        'fp': int(fp),
        'acc': float(acc),
        'prec1': float(prec1),
        'rec1': float(rec1),
        'f11': float(f11),
        'feasible_found': feasible_found,
        'max_acc': float(max_acc),
    }

def select_best(results, min_acc=0.80):
    candidates = [r for r in results if r[2] >= min_acc]
    pool = candidates if candidates else results
    return min(pool, key=lambda r: (r[1], -r[2], -r[4]))

thresholds = np.linspace(0.05, 0.95, 37)
N = 10
results = []

for seed in range(N):
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.25, random_state=seed, stratify=y_train_full
    )
    model_inner = Pipeline(steps=[
        ('preprocess', preprocessor_et),
        ('model', ExtraTreesClassifier(
            random_state=seed,
            n_jobs=-1,
            n_estimators=grid.best_params_['model__n_estimators'],
            max_depth=grid.best_params_['model__max_depth'],
            min_samples_leaf=grid.best_params_['model__min_samples_leaf'],
            max_features=grid.best_params_['model__max_features'],
            class_weight=grid.best_params_['model__class_weight'],
        ))
    ])
    model_inner.fit(X_train, y_train)
    proba_val = model_inner.predict_proba(X_val)[:, 1]
    choice = pick_threshold_min_fp(y_val, proba_val, thresholds, min_acc=0.80)
    best_t = choice['threshold']
    val_note = "VAL_OK" if choice['feasible_found'] else f"VAL_NO(max_acc={choice['max_acc']:.3f})"

    model_final = Pipeline(steps=[
        ('preprocess', preprocessor_et),
        ('model', ExtraTreesClassifier(
            random_state=seed,
            n_jobs=-1,
            n_estimators=grid.best_params_['model__n_estimators'],
            max_depth=grid.best_params_['model__max_depth'],
            min_samples_leaf=grid.best_params_['model__min_samples_leaf'],
            max_features=grid.best_params_['model__max_features'],
            class_weight=grid.best_params_['model__class_weight'],
        ))
    ])
    model_final.fit(X_train_full, y_train_full)
    proba_test = model_final.predict_proba(X_test)[:, 1]
    y_pred = (proba_test >= best_t).astype(int)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1])
    acc = accuracy_score(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    prec1 = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    rec1 = recall_score(y_test, y_pred, pos_label=1)
    f11 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    f1w = f1_score(y_test, y_pred, average='weighted')

    results.append((seed, fp, acc, kappa, prec1, rec1, f11, f1w, best_t, model_final, y_test, y_pred))
    acc_note = "ACC_OK" if acc >= 0.80 else "ACC_NO"
    print(
        f"Seed {seed} → {val_note} | Thr: {best_t:.2f} | {acc_note} Acc: {acc:.4f} | Kappa: {kappa:.4f} | "
        f"Prec(1): {prec1:.4f} | Recall(1): {rec1:.4f} | FP(0→1): {fp} | FN(1→0): {fn}"
    )

best = select_best(results, min_acc=0.80)
best_seed, best_fp, best_acc, best_kappa, best_prec1, best_rec1, best_f11, best_f1w, best_thr, best_model_final, y_test_final, y_pred_final = best

print("\n" + "="*60)
print("MEJOR MODELO FINAL (ExtraTrees + Threshold)")
print("\n" + "="*60)
print(f"Seed: {best_seed}")
print(f"Threshold: {best_thr:.2f}")
print(f"Accuracy: {best_acc:.4f}")
print(f"Kappa: {best_kappa:.4f}")
print(f"Precision (clase=1): {best_prec1:.4f}")
print(f"Recall (clase=1):    {best_rec1:.4f}")
print(f"F1 (clase=1):        {best_f11:.4f}")
print(f"F1 weighted:         {best_f1w:.4f}")
print(f"FP(0→1): {best_fp}")

print("\nClassification Report:\n")
print(classification_report(y_test_final, y_pred_final, digits=3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_final, y_pred_final))

Iniciando GridSearchCV (ExtraTrees) optimizando ACC (y reportando precisión clase 1)...
Fitting 5 folds for each of 16 candidates, totalling 80 fits


c:\Users\samue\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Mejores parámetros:
{'model__class_weight': None, 'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__n_estimators': 300}
Accuracy en CV:        0.8056
Precisión(clase=1) CV: 0.8159

===== ENTRENAMIENTO REPETIDO + TUNING DE UMBRAL (min FP con acc>=0.80) =====
Seed 0 → VAL_OK | Thr: 0.47 | ACC_OK Acc: 0.8057 | Kappa: 0.5400 | Prec(1): 0.7866 | Recall(1): 0.5904 | FP(0→1): 201 | FN(1→0): 514
Seed 1 → VAL_OK | Thr: 0.47 | ACC_OK Acc: 0.8163 | Kappa: 0.5689 | Prec(1): 0.7939 | Recall(1): 0.6231 | FP(0→1): 203 | FN(1→0): 473
Seed 2 → VAL_OK | Thr: 0.50 | ACC_OK Acc: 0.8051 | Kappa: 0.5281 | Prec(1): 0.8233 | Recall(1): 0.5458 | FP(0→1): 147 | FN(1→0): 570
Seed 3 → VAL_NO(max_acc=0.794) | Thr: 0.82 | ACC_NO Acc: 0.7086 | Kappa: 0.1840 | Prec(1): 0.9946 | Recall(1): 0.1466 | FP(0→1): 1 | FN(1→0): 1071
Seed 4 → VAL_OK | Thr: 0.50 | ACC_OK Acc: 0.8062 | Kappa: 0.5337 | Prec(1): 0.8144 | Recall(1): 0.5594 | FP(0→1): 160 | FN(1→0): 553
Seed 5 → VAL_NO(max